# Paso 1 — Catálogo de indicadores del WDI (Banco Mundial)

**TFM:** Inflación en Argentina, un análisis comparativo mediante técnicas de Machine Learning

**Objetivo de este notebook:**
Este notebook solo lista *qué* indicadores existen
dentro de los "topics" (categorías temáticas) del Banco Mundial que se relacionan con
nuestras 4 dimensiones teóricas. Es un paso de exploración: nos sirve para ver el universo
de variables disponibles y decidir, revisando la lista, cuáles tiene sentido conservar
antes de descargar ningún dato real. De acá obtengo la metadata


## Bloque: Importación de librerías

**Objetivo:** cargar las herramientas necesarias para conectarnos a la API del Banco
Mundial y para organizar los resultados en una tabla.


In [1]:
import wbgapi as wb
# wbgapi es la librería de Python que se conecta directamente a la API oficial del
# Banco Mundial (World Bank Open Data). Es la opción más reproducible: en vez de
# bajar un CSV a mano desde una página web, este código pide los datos directamente
# al servidor del Banco Mundial. Cualquiera que corra este mismo notebook va a
# obtener exactamente la misma información.

import pandas as pd

## Bloque: explorar tópicos e indicadores


ver la lista completa de topics disponibles, se puede correr `wb.topic.info()`


In [2]:
wb.topic.info()

id,value
1,Agriculture & Rural Development
2,Aid Effectiveness
3,Economy & Growth
4,Education
5,Energy & Mining
6,Environment
7,Financial Sector
8,Health
9,Infrastructure
10,Social Protection & Labor


In [3]:
# METADATA COMPLETA

import requests
import pandas as pd

BASE_URL = "https://api.worldbank.org/v2/indicator"
params = {
    "format": "json",
    "source": 2,        # 2 = World Development Indicators (WDI)
    "per_page": 20000,  # WDI tiene ~1500 indicadores, con esto entran todos en una página
}

resp = requests.get(BASE_URL, params=params)
resp.raise_for_status()
data = resp.json()

indicators = data[1]  # data[0] = info de paginación, data[1] = lista de indicadores

rows = []
for ind in indicators:
    topics = ind.get("topics") or []
    topic_names = ", ".join(
        t["value"].strip() for t in topics if isinstance(t, dict) and t.get("value")
    )

    rows.append({
        "Topico": topic_names,
        "Indicador": ind.get("name", ""),
        "Description": ind.get("sourceNote", ""),
        "Código WDI": ind.get("id", ""),
    })

metadata = pd.DataFrame(rows)
metadata.head()

,Topico,Indicador,Description,Código WDI
0,Agriculture & Rural Development,Fertilizer consumption (% of fertilizer produc...,Fertilizer consumption measures the quantity o...,AG.CON.FERT.PT.ZS
1,Agriculture & Rural Development,Fertilizer consumption (kilograms per hectare ...,Fertilizer consumption measures the quantity o...,AG.CON.FERT.ZS
2,"Agriculture & Rural Development, Climate Change",Agricultural land (sq. km),Agricultural land refers to the land area that...,AG.LND.AGRI.K2
3,"Agriculture & Rural Development, Climate Chang...",Agricultural land (% of land area),Agricultural land refers to the share of land ...,AG.LND.AGRI.ZS
4,Agriculture & Rural Development,Arable land (hectares),Arable land (in hectares) includes land define...,AG.LND.ARBL.HA


## Bloque: Identificación de códigos WDI duplicados
**Objetivo del bloque:** Detectar qué indicadores del Banco Mundial están repetidos en el dataset de metadatos y conocer su frecuencia.
**Breve explicación:** Se calcula la frecuencia de cada código y se filtran aquellos con más de una aparición para entender el nivel de solapamiento de categorías.

In [5]:
# 1. Calcular la frecuencia de cada código WDI
frecuencia_codigos = metadata["Código WDI"].value_counts()

# 2. Filtrar para quedarnos solo con los que aparecen más de una vez
duplicados = frecuencia_codigos[frecuencia_codigos > 1]

# 3. Mostrar el resultado
print(f"Cantidad de códigos duplicados: {len(duplicados)}")
print("\nCódigos repetidos y su frecuencia:")
print(duplicados)

Cantidad de códigos duplicados: 12

Códigos repetidos y su frecuencia:
Código WDI
GOV_WGI_CC_EST    2
GOV_WGI_CC_SC     2
GOV_WGI_GE_EST    2
GOV_WGI_GE_SC     2
GOV_WGI_PV_EST    2
GOV_WGI_PV_SC     2
GOV_WGI_RL_EST    2
GOV_WGI_RL_SC     2
GOV_WGI_RQ_EST    2
GOV_WGI_RQ_SC     2
GOV_WGI_VA_EST    2
GOV_WGI_VA_SC     2
Name: count, dtype: int64


In [6]:
#QUIERO VER EN DETALLE COMO APARECE ESTA DUPLICACIÓN

# 1. Crear el nuevo DataFrame con todas las filas duplicadas
metadata_duplicados = metadata[metadata["Código WDI"].duplicated(keep=False)].copy()

# 2. Ordenar por Código WDI para agrupar los registros idénticos
metadata_duplicados = metadata_duplicados.sort_values(by="Código WDI")

# 3. Mostrar las primeras filas para inspeccionar el contenido
metadata_duplicados.head(10)

,Topico,Indicador,Description,Código WDI
402,,Control of Corruption - Governance estimate (a...,Control of Corruption (CC) captures perception...,GOV_WGI_CC_EST
403,,Control of Corruption - Governance estimate (a...,Control of Corruption (CC) captures perception...,GOV_WGI_CC_EST
404,,Control of Corruption - Governance score (0-100),Control of Corruption (CC) captures perception...,GOV_WGI_CC_SC
405,,Control of Corruption - Governance score (0-100),Control of Corruption (CC) captures perception...,GOV_WGI_CC_SC
406,,Government Effectiveness - Governance estimate...,Government Effectiveness (GE) captures percept...,GOV_WGI_GE_EST
407,,Government Effectiveness - Governance estimate...,Government Effectiveness (GE) captures percept...,GOV_WGI_GE_EST
408,,Government Effectiveness - Governance score (0...,Government Effectiveness (GE) captures percept...,GOV_WGI_GE_SC
409,,Government Effectiveness - Governance score (0...,Government Effectiveness (GE) captures percept...,GOV_WGI_GE_SC
410,,Political Stability - Governance estimate (app...,Political Stability (PV) captures perceptions ...,GOV_WGI_PV_EST
411,,Political Stability - Governance estimate (app...,Political Stability (PV) captures perceptions ...,GOV_WGI_PV_EST


In [9]:
metadata_duplicados.shape

(24, 4)

## Bloque: Eliminación de códigos WDI duplicados
**Objetivo del bloque:** Limpiar el dataset de metadatos eliminando los registros repetidos según el Código WDI, dejando un único registro por indicador.
**Breve explicación:** Se utiliza `.drop_duplicates()` especificando la columna clave. Se mantiene la primera ocurrencia de cada indicador para garantizar que los códigos sean únicos y queden listos para futuras uniones (merges).

In [10]:
# 1. Guardar la cantidad de filas iniciales para comparar
filas_antes = metadata.shape[0]

# 2. Eliminar duplicados manteniendo la primera aparición
metadata = metadata.drop_duplicates(subset=["Código WDI"], keep="first")

# 3. Verificar el impacto de la limpieza
filas_despues = metadata.shape[0]
print(f"Filas antes de la limpieza: {filas_antes}")
print(f"Filas después de la limpieza: {filas_despues}")
print(f"Total de filas eliminadas: {filas_antes - filas_despues}")

Filas antes de la limpieza: 1510
Filas después de la limpieza: 1498
Total de filas eliminadas: 12


## Bloque: exportación del df con la metadata sin duplicados

In [11]:
metadata.to_excel("metadata.xlsx", index=False)